# Geometric perturbation experiment (interactive)

Run this notebook from the repository root in a HiPerGator Jupyter session with **one GPU**. It generates the images, applies the geometric transforms, inverts them, and plots the results. It is the same experiment as `scripts/run_geometric_registration.slurm`, in one place.

Use a different output directory from the batch job. The default below is `results/geometric_registration/interactive`, so it will not write into `run_001` while that Slurm job is going.

## On HiPerGator Open OnDemand

1. Start a Jupyter session with one GPU on `hpg-turin`, about 8 GB of memory, account `eel5840`.
2. Open this notebook with the project `.conda-env` kernel (the environment built by `scripts/setup_env.slurm`).
3. Set the working directory to the repo, `/home/samuel.cohen/gaussian_rotation`, if the file browser did not already.
4. Run the cells from the top. A finished sample is saved immediately. Re-running the experiment cell continues the same output directory instead of starting over.

`NUM_SAMPLES = 100` matches the batch experiment and takes a few hours. Set it to `2` for a short look at the plots, then put it back to `100` and run the experiment cell again. Keep `SEED` fixed for a given output directory.

## Pipeline

For each sample the notebook draws one Gaussian latent and generates one image. That image is reused for every condition:

```text
z → diffusion → image x → geometric transform → DDIM inversion → z_hat
```

The score is `pearsonr(|z|, |z_hat|)`. Signed correlation, percentile distance, and watermark decoding are not used.

Conditions: no transform; horizontal shifts of 1, 2, 5, and 10 pixels; the same four vertical shifts; rotations of 1°, 2°, 5°, and 10°. New pixels are black. Translations are exact nearest-neighbor shifts, with content moving right or down. Rotations are bicubic, counterclockwise about the center, and stay at the model resolution.

## 1. Configuration

In [ ]:
from pathlib import Path

NUM_SAMPLES = 100
SEED = 42
OUTPUT_DIR = Path("results/geometric_registration/interactive")

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5

## 2. Run the experiment

In [ ]:
import importlib.util
import sys

ROOT = Path.cwd().resolve()
script = ROOT / "scripts" / "run_geometric_registration_experiment.py"
if not script.is_file():
    raise SystemExit(
        f"Cannot find {script}. Open Jupyter with the working directory set to the repository root."
    )
sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("geometric_registration_experiment", script)
geom = importlib.util.module_from_spec(spec)
spec.loader.exec_module(geom)

geom.main(
    [
        "--num-samples", str(NUM_SAMPLES),
        "--seed", str(SEED),
        "--output-dir", str(OUTPUT_DIR),
        "--model-id", MODEL_ID,
        "--num-inference-steps", str(NUM_INFERENCE_STEPS),
        "--device", DEVICE,
        "--dtype", DTYPE,
        "--image-size", str(IMAGE_SIZE),
        "--prompt", PROMPT,
        "--guidance-scale", str(GUIDANCE_SCALE),
    ]
)

## 3. Load the table

The following cells only read `correlations.csv`. Run them again after the experiment cell adds more samples.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else ROOT / OUTPUT_DIR
csv_path = RUN_DIR / "correlations.csv"
if not csv_path.is_file():
    raise SystemExit(f"missing {csv_path}. Run the experiment cell first.")

raw = pd.read_csv(csv_path)
meta = json.loads((RUN_DIR / "metadata.json").read_text(encoding="utf-8"))
order = ["baseline", "horizontal_translation", "vertical_translation", "rotation"]
raw["transformation_type"] = pd.Categorical(raw.transformation_type, categories=order, ordered=True)
print(f"samples {raw.sample_id.nunique()}, rows {len(raw)}, seed {meta.get('seed')}, model {meta.get('model_id')}")
print(f"resolution {meta.get('image_size')}, latent shape {meta.get('latent_shape')}")
print("padding:", meta.get("padding"))

## 4. Summary

Mean, standard deviation, and median of the magnitude correlation. The baseline row is the untransformed image.

In [ ]:
summary = (
    raw.groupby(["transformation_type", "magnitude"], observed=True)["correlation"]
    .agg(mean="mean", std="std", median="median", n="count")
    .reset_index()
)
display(summary.round(4))

## 5. Correlation versus perturbation magnitude

Error bars are ±1 standard error of the mean. The baseline mean is the dashed line.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
baseline_mean = float(summary.loc[summary.transformation_type == "baseline", "mean"].iloc[0])
panels = [
    ("horizontal_translation", "Horizontal translation (px)"),
    ("vertical_translation", "Vertical translation (px)"),
    ("rotation", "Rotation (degrees)"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (kind, title) in zip(axes, panels):
    sub = summary[summary.transformation_type == kind].sort_values("magnitude")
    sem = sub["std"] / np.sqrt(sub["n"])
    ax.errorbar(sub.magnitude, sub["mean"], yerr=sem, fmt="o-", capsize=3, label="mean ± SEM")
    ax.axhline(baseline_mean, color="gray", ls="--", lw=1, label="baseline mean")
    ax.set(xlabel="magnitude", title=title)
axes[0].set_ylabel("correlation of |z| and |z_hat|")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "correlation_vs_magnitude.png", bbox_inches="tight")
plt.show()

## 6. Distribution across samples

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.0), sharey=True)
for ax, (kind, title) in zip(axes, panels):
    block = raw[raw.transformation_type.isin(["baseline", kind])]
    magnitudes = sorted(block.magnitude.unique())
    groups = [block.loc[block.magnitude == m, "correlation"].to_numpy() for m in magnitudes]
    ax.violinplot(groups, positions=magnitudes, widths=0.6 if kind != "rotation" else 0.5, showmedians=True)
    ax.set(xlabel="magnitude", title=title, xticks=magnitudes)
axes[0].set_ylabel("correlation")
fig.tight_layout()
fig.savefig(RUN_DIR / "correlation_violins.png", bbox_inches="tight")
plt.show()

## 7. Matched comparison with the baseline

`delta = correlation(condition) − correlation(baseline)` on the same sample. Negative means that condition correlates less than the untransformed inversion of the same image.

In [ ]:
base = raw.loc[raw.transformation_type == "baseline", ["sample_id", "correlation"]].rename(
    columns={"correlation": "baseline"}
)
matched = raw.merge(base, on="sample_id", how="inner")
matched = matched[matched.transformation_type != "baseline"].copy()
matched["delta"] = matched.correlation - matched.baseline

delta_summary = (
    matched.groupby(["transformation_type", "magnitude"], observed=True)["delta"]
    .agg(mean="mean", std="std", median="median", n="count")
    .reset_index()
)
display(delta_summary.round(4))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (kind, title) in zip(axes, panels):
    sub = delta_summary[delta_summary.transformation_type == kind].sort_values("magnitude")
    sem = sub["std"] / np.sqrt(sub["n"])
    ax.errorbar(sub.magnitude, sub["mean"], yerr=sem, fmt="o-", capsize=3, color="crimson")
    ax.axhline(0, color="k", lw=1)
    ax.set(xlabel="magnitude", title=title)
axes[0].set_ylabel("delta correlation vs baseline")
fig.tight_layout()
fig.savefig(RUN_DIR / "delta_vs_magnitude.png", bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4.0), sharey=True)
for ax, (kind, title) in zip(axes, panels):
    block = matched[matched.transformation_type == kind]
    magnitudes = sorted(block.magnitude.unique())
    groups = [block.loc[block.magnitude == m, "delta"].to_numpy() for m in magnitudes]
    ax.violinplot(groups, positions=magnitudes, widths=0.6 if kind != "rotation" else 0.5, showmedians=True)
    ax.axhline(0, color="k", lw=1)
    ax.set(xlabel="magnitude", title=title, xticks=magnitudes)
axes[0].set_ylabel("delta correlation")
fig.tight_layout()
fig.savefig(RUN_DIR / "delta_violins.png", bbox_inches="tight")
plt.show()

## 8. Example transformations

Sample 0 from this notebook's run: the generated image, a 5 px horizontal shift, a 5 px vertical shift, and a 5° rotation. Black pixels are the fill introduced by the transform.

In [ ]:
examples = [
    ("baseline", RUN_DIR / "examples" / "sample000_baseline.png"),
    ("horizontal 5 px", RUN_DIR / "examples" / "sample000_horizontal_translation_5.png"),
    ("vertical 5 px", RUN_DIR / "examples" / "sample000_vertical_translation_5.png"),
    ("rotation 5°", RUN_DIR / "examples" / "sample000_rotation_5.png"),
]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (title, path) in zip(axes, examples):
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    if path.is_file():
        ax.imshow(plt.imread(path))
    else:
        ax.text(0.5, 0.5, "not saved yet", ha="center", va="center")
fig.tight_layout()
plt.show()

## 9. What this run shows

The printout only describes the table loaded above. A decrease on these samples would mean the unsigned-magnitude correlation moved with the perturbation here. It would not, by itself, establish that the correlation is a reliable registration signal for other models, prompts, or inversions.

In [ ]:
print(f"baseline mean correlation: {baseline_mean:.4f}")
for kind, title in panels:
    sub = summary[summary.transformation_type == kind].sort_values("magnitude")
    means = sub["mean"].to_numpy()
    steps = np.diff(means)
    direction = "non-increasing" if len(steps) and np.all(steps <= 1e-12) else "not monotone"
    deltas = delta_summary[delta_summary.transformation_type == kind].sort_values("magnitude")
    print(f"{title}: mean correlation by magnitude is {direction}")
    print("  magnitudes", sub.magnitude.round(3).tolist())
    print("  means     ", [round(float(v), 4) for v in means])
    print("  mean delta", [round(float(v), 4) for v in deltas["mean"]])